In [8]:
"""
ParkCast SF — Model Training Script
Trains a gradient boosting model to predict parking occupancy %
Logs everything to GCP MLflow and registers the best model.
"""

import os

# ── Fix artifact URI before importing mlflow ──────────────────
os.environ["MLFLOW_ARTIFACT_URI"] = "http://34.133.160.231:5000"


import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://34.133.160.231:5000"   # replace with your GCP IP
EXPERIMENT_NAME     = "parkcast-training"
MODEL_NAME          = "parkcast-occupancy-model"
DATA_PATH           = "data/parkcast_raw.csv"
RANDOM_STATE        = 42

# Features the model uses for prediction
FEATURE_COLS = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_rush_hour",
    "is_street_cleaning",
    "has_nearby_event",
    "is_holiday",
    "is_school_day",
    "is_raining",
    "bad_weather",
    "temperature",
    "total_spaces",
    "neighborhood_encoded",
]

TARGET_COL = "occupancy_pct"

# ── MLflow setup ──────────────────────────────────────────────────────────────
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load and prepare data
# ─────────────────────────────────────────────────────────────────────────────
def load_and_prepare(path):
    print("Loading data...")
    df = pd.read_csv(path)
    print(f"  Loaded {len(df):,} rows, {len(df.columns)} columns")

    # Encode neighborhood as integer
    le = LabelEncoder()
    df["neighborhood_encoded"] = le.fit_transform(df["neighborhood"].astype(str))

    # Keep only feature columns that exist in the dataset
    available_features = [c for c in FEATURE_COLS if c in df.columns]
    missing = [c for c in FEATURE_COLS if c not in df.columns]
    if missing:
        print(f"  Warning: missing features {missing} — will skip them")

    X = df[available_features].fillna(0)
    y = df[TARGET_COL].fillna(0)

    print(f"  Features used: {list(X.columns)}")
    print(f"  Target: {TARGET_COL} (mean={y.mean():.2f}%, std={y.std():.2f}%)")

    return X, y, le, available_features


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Evaluate a model and return metrics
# ─────────────────────────────────────────────────────────────────────────────
def evaluate(model, X_test, y_test):
    preds = model.predict(X_test)
    return {
        "mae":  round(mean_absolute_error(y_test, preds), 4),
        "rmse": round(np.sqrt(mean_squared_error(y_test, preds)), 4),
        "r2":   round(r2_score(y_test, preds), 4),
    }


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Train and log a single model run
# ─────────────────────────────────────────────────────────────────────────────
def train_and_log(model, model_name, params, X_train, X_test, y_train, y_test, features):
    print(f"\nTraining {model_name}...")

    with mlflow.start_run(run_name=model_name):
        # Train
        model.fit(X_train, y_train)

        # Evaluate
        metrics = evaluate(model, X_test, y_test)
        print(f"  MAE:  {metrics['mae']}")
        print(f"  RMSE: {metrics['rmse']}")
        print(f"  R²:   {metrics['r2']}")

        # Log to MLflow
        mlflow.log_params(params)
        mlflow.log_params({"model_type": model_name, "features": str(features)})
        mlflow.log_metrics(metrics)

        # Log model
        # mlflow.sklearn.log_model(
        #     sk_model=model,
        #     artifact_path="model",
        #     registered_model_name=MODEL_NAME,
        #     input_example=X_test.iloc[:1],
        # )
        import os
        import joblib

       # Save model locally on your Mac
        os.makedirs("models", exist_ok=True)
        model_path = f"models/{model_name}.pkl"
        joblib.dump(model, model_path)
        print(f"  Model saved locally to {model_path}")

        run_id = mlflow.active_run().info.run_id
        print(f"  Logged to MLflow run: {run_id}")

        return metrics, run_id


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Promote best model to "champion" alias in registry
# ─────────────────────────────────────────────────────────────────────────────
def promote_best_model(best_run_id):
    print(f"\nPromoting best model (run: {best_run_id}) to champion...")
    client = MlflowClient()

    # Find the model version associated with this run
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    best_version = None
    for v in versions:
        if v.run_id == best_run_id:
            best_version = v.version
            break

    if best_version:
        # Set alias to "champion" so FastAPI can load it by alias
        client.set_registered_model_alias(MODEL_NAME, "champion", best_version)
        print(f"  Model version {best_version} promoted to 'champion'")
    else:
        print("  Could not find model version for best run")

    return best_version


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("ParkCast SF — Model Training")
    print("=" * 60)

    # Load data
    X, y, label_encoder, features = load_and_prepare(DATA_PATH)

    # Train/test split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f"\nTrain size: {len(X_train):,} | Test size: {len(X_test):,}")

    # ── Models to try ─────────────────────────────────────────────────────────
    candidates = [
        {
            "name": "GradientBoosting",
            "model": GradientBoostingRegressor(
                n_estimators=200,
                max_depth=5,
                learning_rate=0.1,
                random_state=RANDOM_STATE,
            ),
            "params": {
                "n_estimators": 200,
                "max_depth": 5,
                "learning_rate": 0.1,
            },
        },
        {
            "name": "RandomForest",
            "model": RandomForestRegressor(
                n_estimators=200,
                max_depth=10,
                random_state=RANDOM_STATE,
            ),
            "params": {
                "n_estimators": 200,
                "max_depth": 10,
            },
        },
        {
            "name": "Ridge",
            "model": Ridge(alpha=1.0),
            "params": {"alpha": 1.0},
        },
    ]

    # ── Train all candidates ──────────────────────────────────────────────────
    results = []
    for candidate in candidates:
        metrics, run_id = train_and_log(
            model=candidate["model"],
            model_name=candidate["name"],
            params=candidate["params"],
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            features=features,
        )
        results.append({
            "name":    candidate["name"],
            "run_id":  run_id,
            "metrics": metrics,
        })

    # ── Pick best model (lowest MAE) ──────────────────────────────────────────
    best = min(results, key=lambda x: x["metrics"]["mae"])
    print("\n" + "=" * 60)
    print("RESULTS SUMMARY")
    print("=" * 60)
    for r in results:
        marker = " ← BEST" if r["name"] == best["name"] else ""
        print(f"  {r['name']:20s} MAE={r['metrics']['mae']:.4f}  "
              f"RMSE={r['metrics']['rmse']:.4f}  "
              f"R²={r['metrics']['r2']:.4f}{marker}")

    # ── Promote best to champion ──────────────────────────────────────────────
    # best_version = promote_best_model(best["run_id"])

    print("\n" + "=" * 60)
    print("TRAINING COMPLETE")
    print("=" * 60)
    print(f"Best model:    {best['name']}")
    print(f"MAE:           {best['metrics']['mae']} %")
    print(f"RMSE:          {best['metrics']['rmse']} %")
    print(f"R²:            {best['metrics']['r2']}")
    # print(f"Model version: {best_version}")
    print(f"Registry name: {MODEL_NAME}")
    print(f"Alias:         champion")
    print(f"\nLoad in FastAPI with:")
    print(f"  mlflow.sklearn.load_model('models:/{MODEL_NAME}@champion')")
    print("=" * 60)


if __name__ == "__main__":
    main()

main()

ParkCast SF — Model Training
Loading data...
  Loaded 21,840 rows, 20 columns
  Features used: ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'is_street_cleaning', 'has_nearby_event', 'is_holiday', 'is_school_day', 'is_raining', 'bad_weather', 'temperature', 'total_spaces', 'neighborhood_encoded']
  Target: occupancy_pct (mean=50.20%, std=27.49%)

Train size: 17,472 | Test size: 4,368

Training GradientBoosting...
  MAE:  7.93
  RMSE: 9.8572
  R²:   0.8687
  Model saved locally to models/GradientBoosting.pkl
  Logged to MLflow run: b3a2f4bee1864bc3ae07ef48110906ea
🏃 View run GradientBoosting at: http://34.133.160.231:5000/#/experiments/6/runs/b3a2f4bee1864bc3ae07ef48110906ea
🧪 View experiment at: http://34.133.160.231:5000/#/experiments/6

Training RandomForest...
  MAE:  7.8833
  RMSE: 9.8037
  R²:   0.8701
  Model saved locally to models/RandomForest.pkl
  Logged to MLflow run: d536e481fff94fbd96d0e52bad9e283c
🏃 View run RandomForest at: http://34.133.160.231:5000/#/e